# FCT Creation Notebook Page
--------------------------------
#### License / Education / Reward & Penalty / Career / Military / Language / Appointment
 - Since System A has separate tables for the 3 companies, the flow is: brz (each company's table) -> slv (merge 3 company tables) -> gld (use the single merged table from slv)
    - Reason for using slv: If we append 3 times for each table, future maintenance and query length become too difficult to manage, so we create a single table in slv.
 - System B performs brz -> gld

--------------------------------------

## FCT License f_hr_license
--------------------------------------
 - System B + System A respectively unioned state
    - Because System B and System A use different logic, if an issue occurs, System B must be modified in the upper query and System A in the lower query.
    - Since columns were created based on System B, some columns might exist in System B but not in System A -> In such cases, System A is processed as NULL.

 - System A only has EMP_NO and no EMP_ID, so EMP_ID is composed of > Alphabet + EMP_NO. In other words, EMP_ID and EMP_NO are the same data.

 - To add a column to the dashboard, it must be added to both System A & System B to be modified.
   - Based on UNION, if the number of query columns and their order do not match, the table cannot be modified.
   - If an error occurs after matching the column count and order, also check CAST -> System B is mostly timestamp type, System A is text type, so a CAST statement must be added.
   - Changing date formats often results in missing data, so it is mandatory to verify if data appears after CAST!

In [ ]:
df = spark.sql("""
-- System B Query: Extract license info by system and integrate into standardized schema
SELECT
LICENSE.MOD_USER_ID  AS MOD_PRSN,  -- Last modifier ID
LICENSE.MOD_DATE AS MOD_DTT,       -- Last modification datetime
LICENSE.TZ_CD AS TZ_CD,            -- Timezone code
LICENSE.TZ_DATE AS TZ_DTT,         -- Timezone based datetime
CAST(LICENSE.EMP_ID AS string ) AS EMP_ID,  -- Unify employee number data type (String)
LICENSE.PHM_LICENSE_ID AS EMP_LCNS_ID,      -- License Unique ID
LICENSE.LICENSE_COMPANY_NM AS ISSUE_ORG,    -- Issuing Institution
LICENSE.LICENSE_TYPE_CD AS LCNS_DIV_CD,     -- License Classification Code
COMM_LICENSE_TY.CD_NM AS LCNS_DIV_NM,       -- License Classification Name (Common Code Join)
COMM_LICENSE.CD_NM AS LCNS_NM,              -- License Name (Common Code Join)
LICENSE.NOTE AS NOTE,                       -- Remarks
LICENSE.BONUS_TYPE AS BONUS_PAY_DIV_CD,     -- Allowance Payment Classification Code
LICENSE.REG_YN AS REG_YN,                   -- Registration Status
LICENSE.LICENSE_NO AS LCNS_NO,              -- License Number
to_timestamp(LICENSE.END_YMD , 'yyyyMMdd') AS VALID_DT,  -- Valid End Date Timestamp Conversion
to_timestamp(LICENSE.STA_YMD , 'yyyyMMdd') AS GET_DT,    -- Acquisition Date Timestamp Conversion
LICENSE.PERSON_ID  AS PRSNL_ID,              -- Personal Identification Number
LICENSE.LICENSE_CD AS LCNS_CD               -- License Code
FROM brz.system_b.PHM_LICENSE LICENSE
LEFT JOIN brz.system_b.common_code COMM_LICENSE ON COMM_LICENSE.CD_KIND ='PHM_LICENSE_CD' AND LICENSE.LICENSE_CD = COMM_LICENSE.CD
LEFT JOIN brz.system_b.common_code COMM_LICENSE_TY ON COMM_LICENSE_TY.CD_KIND ='PHM_LICENSE_TYPE_CD' AND LICENSE.LICENSE_TYPE_CD = COMM_LICENSE_TY.CD
  
UNION  -- Integrate by matching data schemas of System A and System B
-- System A Query: Data extraction using Silver layer (3-company integrated table)
SELECT 
NULL as MOD_PRSN,  -- NULL processing for columns not existing in System A
l.UPDATE_DTS as MOD_DTT,  
NULL as TZ_CD,  
NULL as TZ_DTT,   
CASE  -- Convert System A employee number system to integrated EMP_ID standard (Prefix applied)
 WHEN l.COMPANY_CD = '3000' THEN 'a' || l.EMP_NO -- Company A prefix 'a' assigned
 WHEN l.COMPANY_CD = '2000' THEN 'q' || l.EMP_NO -- Company B prefix 'q' assigned
 WHEN l.COMPANY_CD = '1000' THEN 'f' || l.EMP_NO -- Company C prefix 'f' assigned
  ELSE l.EMP_NO  
 END AS EMP_ID,
NULL as EMP_LCNS_ID,  
l.ISSUE_ISTN_NM as ISSUE_ORG,  
NULL as LCNS_DIV_CD,  
NULL as LCNS_DIV_NM,  
l.QUA_CARD_NM as LCNS_NM,  
l.RMK_DC as NOTE,  
NULL as BONUS_PAY_DIV_CD,  
NULL as REG_YN,  
NULL as LCNS_NO,  
to_timestamp(l.VLID_DT , 'yyyyMMdd') as VALID_DT,  -- Apply same date formatting as System B
to_timestamp(l.ACQS_DT , 'yyyyMMdd') as GET_DT,  
NULL as PRSNL_ID,  
NULL as LCNS_CD  
FROM slv.system_a.hr_license_mst l  -- Reference 3-company integrated view (Silver Layer)
WHERE l.COMPANY_CD in ('1000', '2000', '3000') -- Filter valid affiliate data
""")

In [ ]:
# Load into gld 
df.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_license");

In [ ]:
%%sql
-- Load data from gld into Databricks table
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_license
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_license';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_license

## FCT Education Info f_hr_scholar 
-----------------------------------------
 - System B: Cannot use Final Education Status column [LAST_YN] (since many companies did not indicate education status)
   - Final School Name Logic Rule: A graduation date exists, the school with the latest graduation date is chosen; if graduation dates are the same, the one with the higher education code is set as the final school name.
 - (Added) System B: Major Name [MJR_NM] is fetched from the common table, but if the major name is null, it is replaced by the major name contained in the education fact itself - Oct 23, 2024
   - Originally, the major name fetched from the System B common code was used, but since there is bifurcated data in the source table, it is set to be replaced as above.

 - System A: Final Education column is usable. Set to output the final school name when [LAST_LEDU_YN = 'Y']. 
<br>

In [ ]:
DF = spark.sql("""
-- System B Query: Assign latest/highest education ranking using Window Functions
WITH ranked_scholar AS (  
        SELECT  
            SCH.PERSON_ID,  
            comm.CD_NM AS FIN_SCH_NM,  
            RANK() OVER (  
                PARTITION BY SCH.PERSON_ID  
                ORDER BY SCH.SCH_GRD_CD DESC, COALESCE(SCH.END_YM, '999912') DESC, SCH.STA_YM DESC, comm.CD_NM ASC  
            ) AS rn,  -- Calculate latest education ranking based on degree level and graduation date
            SCH.STA_YM AS STA_YM,  
            SCH.END_YM AS GRAD_DT,  
            SCH.SCH_KIND_CD AS SCH_KIND_CD,  
            SCH.SCH_GRD_CD AS SCH_GRD_CD  
        FROM brz.system_b.PHM_SCHOLAR SCH  
        LEFT JOIN brz.system_b.common_code comm ON comm.cd_kind = 'PHM_SCH_CD' AND sch.SCH_CD = comm.cd  
        WHERE SCH.END_YM IS NOT NULL 
    )  
    , final_scholar AS (  
        SELECT  
            PERSON_ID,  
            FIN_SCH_NM,  
            SCH_KIND_CD,  
            MAX(STA_YM) AS MAX_STA_YM,  
            MAX(GRAD_DT) AS MAX_GRAD_DT,  
            MAX(SCH_GRD_CD) AS MAX_SCH_GRD_CD  
        FROM ranked_scholar  
        WHERE rn = 1  -- Extract only the top rank data to specify final education
        GROUP BY PERSON_ID, FIN_SCH_NM, SCH_KIND_CD  
    )  
SELECT DISTINCT 
SCH.SUB_MAJOR_NM AS SUBMJR_NM,
SCH.SUB_MAJOR_CD AS SUBMJR_CD,
SCH.ENTRANCE_CD AS SCHENT_DIV_CD,
SCH.SCH_KIND_CD AS SCH_DIV_CD,
SCH.DAY_KIND_CD AS DAY_NIGHT_DIV_CD,
SCH.SCH_PLACE_CD AS SCH_PLC_CD,
SCH.SCH_PLACE_NM AS SCH_PLC_NM,
SCH.SCH_PLACE_DET_CD AS SCH_PLC_DTLS_CD,
SCH.SCH_PLACE_DET_NM AS PLC_REGION_DTLS,
SCH.LAST_YN AS LAST_SCHLST_YN,
SCH.CAREER_NUM AS CAREER_ADMIT_MTH_CNT,
SCH.NOTE AS NOTE,
SCH.MOD_USER_ID AS MOD_PRSN,
SCH.MOD_DATE AS MOD_DTT,
SCH.TZ_CD AS TZ_CD,
SCH.TZ_DATE AS TZ_DTT,
SCH.PHM_SCHOLAR_ID AS EMP_SCHLST_ID,
CAST(SCH.EMP_ID AS varchar(20)) AS EMP_ID,
SCH.PERSON_ID AS PRSNL_ID,
SCH.STA_YM AS SCHENT_YYYYMM,
SCH.END_YM AS GRAD_YYYYMM,
dsch.DTL_CD AS SCHLST_CD, -- Education Level Code
SCH.SCH_CD AS SCH_CD,
cast(SCH.MAJOR_CD as VARCHAR(20)) AS MAJOR_CD,
SCH.DOU_MAJOR_CD AS DOU_MAJOR_CD,
SCH.DOU_MAJOR_NM AS DOU_MAJOR_NM,
dsch.DTL_NM   AS SCHLST_NM, 
cast(comm.CD_NM AS VARCHAR(40)) AS SCH_NM,
coalesce(list.CD_NM, SCH.MAJOR_NM) AS MJR_NM, -- Major Name Standardization: Apply common code first
CASE  
        WHEN comm.CD_NM = fs.FIN_SCH_NM AND (SCH.END_YM = fs.MAX_GRAD_DT OR SCH.END_YM IS NULL) AND SCH.STA_YM = fs.MAX_STA_YM AND SCH.SCH_GRD_CD = fs.MAX_SCH_GRD_CD AND (fs.MAX_SCH_GRD_CD >= '01' OR list.CD_NM IS NOT NULL)  
        THEN fs.FIN_SCH_NM  
        ELSE NULL  
    END AS LAST_SCH_NM -- Final school name mapping based on business logic
FROM brz.system_b.PHM_SCHOLAR SCH   
LEFT JOIN brz.system_b.common_code comm ON comm.cd_kind = 'PHM_SCH_CD' AND sch.SCH_CD = comm.cd     
LEFT JOIN brz.system_b.common_code list  ON list.cd_kind = 'PHM_MAJOR_CD' AND sch.MAJOR_CD = list.cd 
LEFT JOIN ref.hr.code_master dsch ON dsch.df_cd = 'LaSchRatCl' AND sch.SCH_GRD_CD = dsch.Value1 
left join gld.hr.d_hr_school grad on sch.SCH_GRD_CD = grad.SCHL_CD  
LEFT JOIN final_scholar fs ON fs.PERSON_ID = SCH.PERSON_ID

UNION
-- System A Query: Match System B schema and convert employee number prefix
SELECT 
NULL AS SUBMJR_NM,
NULL AS SUBMJR_CD,
NULL AS SCHENT_DIV_CD,
NULL AS SCH_DIV_CD,
NULL AS DAY_NIGHT_DIV_CD,
NULL AS SCH_PLC_CD,
NULL AS SCH_PLC_NM,
NULL AS SCH_PLC_DTLS_CD,
NULL AS PLC_REGION_DTLS,
sch.LAST_LEDU_YN as LAST_SCHLST_YN,
NULL AS CAREER_ADMIT_MTH_CNT,
NULL AS NOTE,
NULL AS MOD_PRSN,
sch.UPDATE_DTS as MOD_DTT,
NULL AS TZ_CD,
NULL AS TZ_DTT,
NULL AS EMP_SCHLST_ID,
CASE
WHEN sch.COMPANY_CD = '3000' THEN 'a' || sch.EMP_NO 
WHEN sch.COMPANY_CD = '2000' THEN 'q' || sch.EMP_NO 
WHEN sch.COMPANY_CD = '1000' THEN 'f' || sch.EMP_NO 
ELSE sch.EMP_NO
END AS EMP_ID, -- Generate enterprise integrated employee number via company-specific prefixes
NULL AS PRSNL_ID,
TO_CHAR(TO_DATE(sch.MTRC_DT, 'yyyyMMdd'), 'yyyyMM') AS SCHENT_YYYYMM,  -- Admission Date Data Formatting
TO_CHAR(TO_DATE(sch.GRDT_DT, 'yyyyMMdd'), 'yyyyMM') AS GRAD_YYYYMM,    -- Graduation Date Data Formatting
COALESCE(dicd.DTL_CD, docd.DTL_CD, stcd.DTL_CD, NULL ) AS SCHLST_CD,
sch.SCHL_CD as SCH_CD,
cast (sch.DMJ_CD as VARCHAR(20)) as MAJOR_CD,
sch.PL_MJR_CD as DOU_MAJOR_CD,
sch.PL_MJR_NM as DOU_MAJOR_NM,
COALESCE(dicd.DTL_NM, docd.DTL_NM, stcd.DTL_NM, NULL ) AS SCHLST_NM, 
cast (sch.SCHL_NM  as VARCHAR(40))as SCH_NM,
sch.MJR_NM as MJR_NM,
CASE
WHEN sch.LAST_LEDU_YN = 'Y' THEN SCHL_NM -- Output school name based on final education status ('Y')
ELSE NULL
END AS LAST_SCH_NM
from slv.system_a.HR_SCHOCARE_MST sch
LEFT JOIN (
    SELECT DISTINCT DTL_CD, DTL_NM, Value7 FROM ref.hr.code_master WHERE df_cd = 'LaSchRatCl' AND Value7 IS NOT NULL
    ) dicd ON dicd.Value7 = sch.LEDU_CD AND sch.COMPANY_CD = '3000' 
LEFT JOIN (
    SELECT DISTINCT DTL_CD, DTL_NM, Value3 FROM ref.hr.code_master WHERE df_cd = 'LaSchRatCl' AND Value3 IS NOT NULL
    ) docd ON docd.Value3 = sch.LEDU_CD AND sch.COMPANY_CD = '1000' 
LEFT JOIN (
    SELECT DISTINCT DTL_CD, DTL_NM, Value5 FROM ref.hr.code_master WHERE df_cd = 'LaSchRatCl' AND Value5 IS NOT NULL
    ) stcd ON stcd.Value5 = sch.LEDU_CD AND sch.COMPANY_CD = '2000' 
where COMPANY_CD in ('1000', '2000' , '3000')
""")

In [ ]:
DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_scholar");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_scholar
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_scholar';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_scholar

## FCT Reward & Penalty Info f_hr_reward_penalty
-----------------------------------------
 - System B: Discipline & Reward exist in a single table [brz.system_b.PPM_MNT]
 - System A: Discipline & Reward exist in different tables [Reward: SLV.system_a.hr_huprize_list / Discipline: SLV.system_a.HR_HUDISCIP_LIST]
 - Ultimately, UNION is performed 3 times (System B Reward/Penalty + System A Reward + System A Discipline)

In [ ]:
DF = spark.sql("""
-- System B Query
SELECT 
MNT.APPLY_YN AS CAM_APPLY_YN,
MNT.DEL_YN AS CAM_DEL_YN,
MNT.COMPANY_CD AS HR_FIELD_CD,
CAST(MNT.EMP_ID AS STRING) AS EMP_ID,
MNT.TYPE_CD AS RWD_RNTY_DIV_CD,
MNT.KIND_CD AS RWD_RNTY_KIND_CD,
MNT.INOFF_CD AS INOUT_HOUSE_DIV_CDD,
MNT.PPM_NO AS RWD_RNTY_NO,
cls.CD_NM AS RWD_RNTY_NM, -- Type (Reward/Penalty)
MNT.PPM_YMD AS RWD_RNTY_DT, -- Reward/Penalty Date
MNT.PPM_MON AS RWD_AMT,
MNT.PPM_DESC AS RWD_PNTY_RSN,
MNT.END_YMD AS END_DT,
MNT.CUST_COL1 AS PERCOM_DT_FRST,
MNT.INPUT_TYPE_CD AS INPUT_DATA_TYPE_CD,
MNT.MNT_POOL_ID AS MENTOR_POOL_ID,
MNT.CAM_YN AS CAM_REG_YN,
MNT.CAM_DATE AS CAM_LINK_DTT,
MNT.CAM_DOC_ID AS CAM_CNF_DOC_ID,
MNT.CAM_PRE_ID AS CAM_RCPNT_ID,
MNT.NOTE AS NOTE,
MNT.MOD_USER_ID AS MOD_PRSN,
MNT.MOD_DATE AS MOD_DTT,
MNT.TZ_CD AS TZ_CD,
MNT.TZ_DATE AS TZ_DTT,
MNT.PPM_MNT_ID AS RWD_DTLS_MNGMNT_ID
FROM brz.system_b.PPM_MNT MNT 
LEFT JOIN brz.system_b.common_code cls ON cls.CD_KIND = 'PPM_TYPE_CD' AND MNT.TYPE_CD= cls.CD

UNION
-- System A Query
-- Reward 
SELECT  
NULL AS CAM_APPLY_YN, 
NULL AS CAM_DEL_YN, 
NULL AS HR_FIELD_CD, 
CASE -- Employee Number 
    WHEN COMPANY_CD = '3000' THEN 'a' || EMP_NO -- Company A
    WHEN COMPANY_CD = '2000' THEN 'q' || EMP_NO -- Company B
    WHEN COMPANY_CD = '1000' THEN 'f' || EMP_NO -- Company C
    ELSE EMP_NO  
END AS EMP_ID,  
NULL AS RWD_RNTY_DIV_CD, 
NULL AS RWD_RNTY_KIND_CD, 
NULL AS INOUT_HOUSE_DIV_CDD,
NULL AS RWD_RNTY_NO, 
'Reward' AS RWD_RNTY_NM, -- Reward Name 
TO_TIMESTAMP(PRZ_DT  , 'yyyyMMdd') AS RWD_RNTY_DT, -- Reward Datetime 
NULL AS RWD_AMT, 
PRZ_DC AS RWD_PNTY_RSN, -- Reward Details 
NULL AS END_DT, 
NULL AS PERCOM_DT_FRST, 
NULL AS INPUT_DATA_TYPE_CD,  
NULL AS MENTOR_POOL_ID, 
NULL AS CAM_REG_YN,
NULL AS CAM_LINK_DTT, 
NULL AS CAM_CNF_DOC_ID, 
NULL AS CAM_RCPNT_ID,
NULL AS NOTE, 
NULL AS MOD_PRSN, 
UPDATE_DTS AS MOD_DTT,
NULL AS TZ_CD,
NULL AS TZ_DTT, 
NULL AS RWD_DTLS_MNGMNT_ID
FROM SLV.system_a.hr_huprize_list

UNION
-- Discipline 
SELECT  
NULL AS CAM_APPLY_YN, 
NULL AS CAM_DEL_YN, 
NULL AS HR_FIELD_CD, 
CASE -- Employee Number 
    WHEN COMPANY_CD = '3000' THEN 'a' || EMP_NO -- Company A
    WHEN COMPANY_CD = '2000' THEN 'q' || EMP_NO -- Company B
    WHEN COMPANY_CD = '1000' THEN 'f' || EMP_NO -- Company C
    ELSE EMP_NO  
END AS EMP_ID,  
NULL AS RWD_RNTY_DIV_CD, 
NULL AS RWD_RNTY_KIND_CD, 
NULL AS INOUT_HOUSE_DIV_CDD,
NULL AS RWD_RNTY_NO, 
'Discipline' AS RWD_RNTY_NM, -- Reward/Penalty Name 
TO_TIMESTAMP(DCPL_DT , 'yyyyMMdd') AS RWD_RNTY_DT, -- Reward/Penalty Datetime 
NULL AS RWD_AMT, 
DCPL_DC AS RWD_PNTY_RSN, -- Reward/Penalty Details 
NULL AS END_DT, 
NULL AS PERCOM_DT_FRST, 
NULL AS INPUT_DATA_TYPE_CD,  
NULL AS MENTOR_POOL_ID, 
NULL AS CAM_REG_YN,
NULL AS CAM_LINK_DTT, 
NULL AS CAM_CNF_DOC_ID, 
NULL AS CAM_RCPNT_ID,
NULL AS NOTE, 
NULL AS MOD_PRSN, 
UPDATE_DTS AS MOD_DTT,
NULL AS TZ_CD,
NULL AS TZ_DTT, 
NULL AS RWD_DTLS_MNGMNT_ID
FROM SLV.system_a.HR_HUDISCIP_LIST;
""")

In [ ]:
DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_reward_penalty");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_reward_penalty
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_reward_penalty';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_reward_penalty

## FCT Career Info f_hr_career
-----------------------------------------
 - No special notes for both System B & System A
<br>

In [ ]:
DF = spark.sql("""
SELECT  
-- System B Query 
    PHM_CAREER_ID AS EMP_CAREER_ID,  
    PLACE_NM AS PLC,  
    PERSON_ID AS PRSNL_ID,  
    STA_YMD AS WORK_STRT_DT,  
    END_YMD AS WORK_END_DT,  
    ORG_CORP_NM AS WORK_ORG_NM,  
    POSITION_NM AS LAST_JOB_GD_NM,  
    RETIRE_CAUSE AS RETRD_RSN,  
    RCAREER_NUM AS REAL_CAREER_MTH_CNT,  
    RECO_RATE AS ADMIT_RATE,  
    CAST(CAREER_NUM AS varchar(20)) AS ADMIT_CAREER_MTH_CNT,  
    NOTE AS NOTE,  
    MOD_USER_ID AS MOD_PRSN,  
    MOD_DATE AS MOD_DTT,  
    TZ_CD AS TZ_CD,  
    TZ_DATE AS TZ_DTT,  
    ORG_ENG_NM AS EMGL_CMP_NM,  
    CAST(EMP_ID AS varchar(20)) AS EMP_ID,
    WORK_NM 
FROM  brz.system_b.phm_career  
  
UNION  
  -- System A Query
SELECT  
    NULL AS EMP_CAREER_ID,  
    NULL AS PLC,  
    NULL AS PRSNL_ID,  
   to_timestamp(JNCO_DT, 'yyyyMMdd') AS WORK_STRT_DT,  
   to_timestamp(RETR_DT, 'yyyyMMdd') AS WORK_END_DT,  
    COMPANY_NM AS WORK_ORG_NM,  
    NULL AS LAST_JOB_GD_NM,  
    NULL AS RETRD_RSN,  
    NULL AS REAL_CAREER_MTH_CNT,  
    NULL AS ADMIT_RATE,  
   CAST(CAST(ROUND(MONTHS_BETWEEN(to_date(RETR_DT, 'yyyyMMdd'), to_date(JNCO_DT, 'yyyyMMdd')), 0) AS INT) AS VARCHAR(20)) AS ADMIT_CAREER_MTH_CNT,
    NULL AS NOTE,  
    NULL AS MOD_PRSN,  
    UPDATE_DTS as MOD_DTT,  
    NULL AS TZ_CD,  
    NULL AS TZ_DTT,  
    NULL AS EMGL_CMP_NM,  
    CASE  
        WHEN COMPANY_CD = '3000' THEN 'a' || EMP_NO -- Company A
        WHEN COMPANY_CD = '2000' THEN 'q' || EMP_NO -- Company B
        WHEN COMPANY_CD = '1000' THEN 'f' || EMP_NO -- Company C
        ELSE EMP_NO  
    END AS EMP_ID,
    RSPT_TSK_NM
FROM  slv.system_a.hr_career_mst  
WHERE COMPANY_CD in ('1000', '2000', '3000')
""")

In [ ]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY") # Configured because date parsing changed from Spark 3.0, which may cause errors

DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_career");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_career
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_career';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_career

## FCT Military Service Info f_hr_military
-----------------------------------------
 - No special notes for both System B & System A
<br>

In [ ]:
DF = spark.sql("""
 -- System B Query
SELECT 
 cast(ARMY.EMP_ID as VARCHAR(20)) AS EMP_ID
,ARMY.ARMY_NO_CD AS INCMPL_RSN
,ARMY.ARMY_SERV_CD AS SRV_TYPE_CD
,ARMY.PHM_ARMY_ID AS EMP_MIL_SRVC_ID
,ARMY.ARMY_NO_REASON_CD AS MIL_CPLT_DIV_CD
,ARMY.ARMY_TYPE_CD AS MIL_TYPE_CD
,typ.CD_NM AS MIL_TYPE_NM -- Military Branch Name
,ARMY.ARMY_BRANCH_CD AS MIL_BRNCH_CD
,ARMY.ARMY_CLASS_CD AS MIL_RANK_CD
,cla.CD_NM AS MIL_RANK_NM -- Rank Name
,ARMY.ARMY_NO AS MIL_NO
,ARMY.ARMY_DISCHARGE_CD AS DSCHRG_DIV_CD
,ARMY.IN_YMD AS MIL_JOIN_DT
,ARMY.OUT_YMD AS DSCHRG_DT
,ARMY.PLUS_MM AS WRKPLC_RSRV_ARMY_JOIN_YN
,ARMY.ARMY_CLASSIFICATION_CD AS MIL_CLASS_CD
,cls.CD_NM AS MIL_CLASS_NM -- Military Service Class Name
,ARMY.ARMY_MTALENT_CD AS MIL_SP_CD
,mtl.CD_NM AS MIL_SP_NM -- Main Specialty Name
,ARMY.NOTE AS NOTE
,ARMY.MOD_USER_ID AS MOD_PRSN
,ARMY.MOD_DATE AS MOD_DTT
,ARMY.TZ_CD AS TZ_CD
,ARMY.TZ_DATE AS TZ_DTT
,ARMY.PERSON_ID AS PRSNL_ID
,ROUND(MONTHS_BETWEEN(TO_DATE(DSCHRG_DT, 'yyyy-MM-dd'), TO_DATE(MIL_JOIN_DT, 'yyyy-MM-dd'))) AS MIL_ADMIT_MTH_CNT
FROM brz.system_b.PHM_ARMY ARMY
LEFT JOIN brz.system_b.common_code cls ON cls.CD_KIND ='PHM_ARMY_CLASSIFICATION_CD' and army.ARMY_CLASSIFICATION_CD = cls.cd -- Military Service Class
LEFT JOIN brz.system_b.common_code mtl ON mtl.CD_KIND ='PHM_ARMY_MTALENT_CD' and army.ARMY_MTALENT_CD = mtl.cd -- Main Specialty Name
LEFT JOIN brz.system_b.common_code typ ON typ.CD_KIND ='PHM_ARMY_TYPE_CD' and army.ARMY_TYPE_CD = typ.cd -- Military Branch
LEFT JOIN brz.system_b.common_code cla ON cla.CD_KIND ='PHM_ARMY_CLASS_CD' and army.ARMY_CLASS_CD = cla.cd -- Rank


union 
-- System A Query 
SELECT DISTINCT
    CASE
        WHEN f.COMPANY_CD = '3000' THEN 'a' || f.EMP_NO -- Company A
        WHEN f.COMPANY_CD = '2000' THEN 'q' || f.EMP_NO -- Company B
        WHEN f.COMPANY_CD = '1000' THEN 'f' || f.EMP_NO -- Company C
        ELSE f.EMP_NO
    END AS EMP_ID, -- Employee ID
    f.EXMT_REASON_CD AS INCMPL_RSN,
    null as SRV_TYPE_CD,
    null as EMP_MIL_SRVC_ID,
    null as MIL_CPLT_DIV_CD,
    f.AMFT_CD AS MIL_TYPE_CD, -- Military Branch
    type.SYSDEF_NM AS MIL_TYPE_NM, -- Common Code Table / HR/P00770
    null as MIL_BRNCH_CD, 
    f.MLH_CD AS MIL_RANK_CD, -- Rank Code
    rank.SYSDEF_NM AS MIL_RANK_NM, -- Common Code Table / HR/P00800
    null as MIL_NO, 
    null as DSCHRG_DIV_CD, 
    TO_TIMESTAMP(f.ENST_DT, 'yyyyMMdd') AS MIL_JOIN_DT, -- Enlistment Date
    TO_TIMESTAMP(f.LAYOFF_DT, 'yyyyMMdd') AS DSCHRG_DT, -- Discharge Date
    NULL AS WRKPLC_RSRV_ARMY_JOIN_YN, 
    f.COSS_CD AS MIL_CLASS_CD, -- Military Service Class Code
    clas.SYSDEF_NM AS MIL_CLASS_NM, -- Common Code Table / HR/P06990
    NULL AS MIL_SP_CD, 
    NULL AS MIL_SP_NM, 
    NULL AS NOTE, 
    NULL AS MOD_PRSN, 
    f.UPDATE_DTS AS MOD_DTT, 
    NULL AS TZ_CD, 
    NULL AS TZ_DTT, 
    NULL AS PRSNL_ID, 
    ROUND(MONTHS_BETWEEN(to_timestamp(f.LAYOFF_DT, 'yyyyMMdd'), to_timestamp(f.ENST_DT, 'yyyyMMdd'))) AS MIL_ADMIT_MTH_CNT
FROM slv.system_a.HR_EMPINFO_DTL f 
LEFT JOIN slv.system_a.ma_codedtl type ON type.SYSDEF_CD = f.AMFT_CD AND type.MODULE_CD = 'HR' AND type.FIELD_CD ='P00770' AND type.COMPANY_CD IN ('1000', '2000', '3000')
LEFT JOIN slv.system_a.ma_codedtl rank ON rank.SYSDEF_CD = f.AMFT_CD AND rank.MODULE_CD = 'HR' AND rank.FIELD_CD ='P00800' AND rank.COMPANY_CD IN ('1000', '2000', '3000')
LEFT JOIN slv.system_a.ma_codedtl clas ON clas.SYSDEF_CD = f.AMFT_CD AND clas.MODULE_CD = 'HR' AND clas.FIELD_CD ='P06990' AND clas.COMPANY_CD IN ('1000', '2000', '3000')
WHERE f.company_cd IN ('1000', '2000', '3000'); 
""")

In [ ]:
DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_military");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_military
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_military';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_military

## FCT Language Info f_hr_language
-----------------------------------------
 - No special notes for both System B & System A
<br>

In [ ]:
DF = spark.sql("""
-- System B Query
SELECT 
est.PHM_LANG_EST_ID AS EMP_LANG_ESTM_ID -- Employee Language Evaluation ID 
,CAST(est.EMP_ID as varchar(20)) AS EMP_ID -- Employee ID 
,est.PERSON_ID AS PRSNL_ID -- Person ID 
,est.LANG_CD AS LANG_CD -- Language Code [PHM_LANG_CD] 
,LANG.cd_nm  AS LANG_NM
,est.STD_YY AS STD_YYYY -- Year 
,est.EST_YMD AS ESTM_DT-- Evaluation Date 
,est.VALID_YMD AS VALID_DT -- Expiration Date 
,est.EST_CD AS LANG_DIV_CD -- Language Classification Code [PHM_EST_CD] 
,estc.cd_nm AS LANG_DIV_NM -- Language Classification Name
,est.EST_ORG_CD AS ESTM_ORG_CD -- Evaluation Agency Code [PHM_EST_ORG_CD] 
,ORG.cd_nm AS EST_ORG_NM -- Evaluation Agency Name
,est.EST_PNT AS LANG_SCR -- Language Score, Functional level 
,est.EST_GRD_CD AS LANG_GRD_CD -- Language Grade Code [PHM_EST_GRD_CD] 
,GRD.cd_nm AS LANG_GRD_NM -- Language Grade Name
,est.INTERNAL_PNT AS PRSNT_LEV -- Present Level 
,est.NOTE -- Remarks
,est.MOD_USER_ID AS MOD_PRSN
,est.MOD_DATE AS MOD_DTT
,est.TZ_CD AS TZ_CD
,est.TZ_DATE AS TZ_DTT
FROM brz.system_b.PHM_LANG_EST est 
LEFT JOIN brz.system_b.common_code LANG ON LANG.CD_KIND ='PHM_LANG_CD' and est.LANG_CD = LANG.cd -- Language Code 
LEFT JOIN brz.system_b.common_code estc ON estc.CD_KIND ='PHM_EST_CD' and est.EST_CD = estc.cd -- Language Classification Code [PHM_EST_CD] 
LEFT JOIN brz.system_b.common_code ORG ON ORG.CD_KIND ='PHM_EST_ORG_CD' and est.EST_ORG_CD = ORG.cd -- Evaluation Agency Code [PHM_EST_ORG_CD] 
LEFT JOIN brz.system_b.common_code GRD ON GRD.CD_KIND ='PHM_EST_GRD_CD' and est.EST_GRD_CD = GRD.cd -- Language Grade Code [PHM_EST_GRD_CD] 

union 

SELECT 
-- System A Query
    NULL AS EMP_LANG_ESTM_ID, 
    CASE
        WHEN f.COMPANY_CD = '3000' THEN 'a' || f.EMP_NO -- Company A
        WHEN f.COMPANY_CD = '2000' THEN 'q' || f.EMP_NO -- Company B
        WHEN f.COMPANY_CD = '1000' THEN 'f' || f.EMP_NO -- Company C
        ELSE f.EMP_NO
    END AS EMP_ID, -- Employee ID
    NULL AS PRSNL_ID,
    f.LANG_CD, -- Language Code
    m.SYSDEF_NM AS LANG_NM, -- Language Name
    NULL AS STD_YYYY,
    TO_timestamp(f.APYEXM_DT, 'yyyyMMdd') AS ESTM_DT, -- Evaluation Date
    TO_timestamp(f.VLID_DT ,'yyyyMMdd') AS VALID_DT, -- Expiration Date
    f.LANG_FG_CD AS LANG_DIV_CD, -- Language Classification Code
    fg.SYSDEF_NM AS LANG_DIV_NM, -- Language Classification Name
    f.TRIAL_ISTN_CD AS ESTM_ORG_CD, -- Evaluation Agency Code
    f.TRIAL_ISTN_NM AS EST_ORG_NM, -- Evaluation Agency Name
    f.LANG_PT AS LANG_SCR, -- Language Score
    f.LANG_GRD_CD AS LANG_GRD_CD, -- Language Grade Code
    f.LANG_GRD_NM AS LANG_GRD_NM, -- Language Grade Name
    NULL AS PRSNT_LEV,
    f.RMK_DC AS NOTE, 
    NULL AS MOD_PRSN,
    f.UPDATE_DTS AS MOD_DTT,
    NULL AS TZ_CD, 
    NULL AS TZ_DTT 
FROM slv.system_a.hr_forelang_mst f 
LEFT JOIN slv.system_a.ma_codedtl m ON m.SYSDEF_CD = f.lang_cd AND m.MODULE_CD = 'HR' AND m.FIELD_CD ='Z001_20343' AND m.COMPANY_CD IN ('1000', '2000', '3000')
LEFT JOIN slv.system_a.ma_codedtl fg ON fg.SYSDEF_CD = f.lang_cd AND fg.MODULE_CD = 'HR' AND fg.FIELD_CD ='P00930' AND fg.COMPANY_CD IN ('1000', '2000', '3000')
where f.COMPANY_CD in ( '1000' , '2000' , '3000');
""")

In [ ]:
DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_language");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_language
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_language';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_language

## FCT Appointment Info f_hr_appoint_history
--------------------------
- Appointment Info = System B Appointment + System B Manual Appointment + System A Appointment (Total 3 table union)
- System B Appointment: brz.system_b.CAM_HISTORY  /  System B Manual Appointment: BRZ.system_b.cam_history_bef  /  System A Appointment: slv.system_a.cam_past
- System A: The relevant past appointment slv.system_a.cam_past table is processed via query at the brz level and brought in
- System A: For the Rank column, use **POSI_CD [Position]**!!  /   _!![PSTN_CD NOT used]!! _
- Voluntary / Involuntary: In case of resignation, System A does not entirely represent Voluntary/Involuntary (Voluntary/Involuntary Master: Only EMP_NO existing in the HR_EMP_SDTL table can be classified; others cannot be classified)

In [ ]:
DF = spark.sql("""
SELECT DISTINCT  -- System B Appointment
cam.REN_YMD AS LAYOFF_RTRN_SCHDL_DT
,cam.DIS_YN AS DSPTCH_YN
,cam.DIS_KIND_CD AS DSPTCH_DIV_CD
,cam.DIS_RTN_YMD AS DSPTCH_RTRN_SCHDL_DT
,cam.DIS_ORG_ID AS DSPTCH_DEPT_CD
,cam.DIS_AREA_NM AS DSPTCH_PLC_NM
,cam.HIRE_CD AS HIRED_DIV_CD
,cam.PUNISH_END_YMD AS PENAL_CNCL_SCHDL_DT
,cam.MAS_YN AS HR_RFLCTN_YN
,cam.PAY_DUTY_CD AS SLRY_JOB_TTLE_GRD
,cam.PRINT_YN AS PRNT_YN
,cam.CUST_COL2 AS JOB_GRP_CD
,cam.CUST_COL3 AS RFRNC_DIV_CD
,cam.CUST_COL4 AS ENTRST_YN
,cam.CUST_COL5 AS ENTRST_END_SCHDL_DT
,cam.CUST_COL7 AS DEL_YN
,cam.CUST_COL8 AS CNF_DOC_NO
,cam.CUST_COL9 AS CNF_DOC_NM
,cam.CUST_COL10 AS CAM_RSN
,cam.NOTE AS NOTE
,cam.MOD_USER_ID AS MOD_PRSN
,cam.MOD_DATE AS MOD_DTT
,cam.TZ_CD AS TZ_CD
,cam.TZ_DATE AS TZ_DTT
,cam.SUB_COMPANY_CD AS CMP_DIV_CD -- Company Code
,com.DTL_NM AS CMP_DIV_NM -- Company Name
,cam.PRO_GRADE_CD AS SLRY_STRT_SLRY_GRD_CD
,cam.PRO_KIND_CD AS FIRST_APPT_SLRY_KIND_CD
,cam.CNT_AMT AS YRLY_SLRY_AMT
,cam.SEND_YN AS JOB_SCND_YN
,cam.SEND_SUB_COMPANY_CD AS JOB_SCND_CAM_CMP_CD
,cam.SEND_JOP_CD AS JOB_SCND_JOB_GRP_CD
,cam.RETIRE_TYPE_CD AS RETRD_RSN_CD
,retire.CD_NM AS RETRD_RSN_NM
,cam.PAY_GRADE AS SLRY_GRD
,cam.PAY_JOP_CD AS SLRY_JOB_GRP
,cam.NEXT_POS_YMD AS NEXTERM_JOB_GD_PRMT_DT
,cam.CAM_HISTORY_ID AS CAM_HIST_ID
,CAST(cam.EMP_ID AS STRING) AS EMP_ID
,cam.PERSON_ID AS PRSNL_ID
,cam.CAM_DOC_ID AS CAM_CNF_ID
,cam.STA_YMD AS CAM_STRT_DT
,cam.END_YMD AS CAM_END_DT
,cam.CAM_YMD AS CAM_DT -- Appointment Date
,cam.SEQ AS CAM_ORDER
,cam.TYPE_CD AS CAM_CD
,COALESCE(mre.dtl_cd ,cam.CAU_CD , null) AS CAM_CLASS_CD
,cam.COMPANY_CD AS CAM_HR_ZONE_CD
,cam.ORG_ID AS CAM_DEPT_ID
,cam.POS_CD as CAM_JOB_RANK_CD -- Rank Code 
,rank.CD_NM as CAM_JOB_RANK_NM -- Rank Name
,cam.JOB_CD AS JOB_CD
,cam.DUTY_CD as CAM_JOB_TTLE_CD -- Title Code 
,duty.cd_nm as CAM_JOB_TTLE_NM -- Title Name
,cam.EMP_KIND_CD AS CAM_EMP_KIND_CD
,cam.PAY_KIND_CD AS CAM_YRLY_SLRY_TYPE
,cam.PROBATION_YN AS PRBTN_YN
,cam.PROBATION_END_YMD AS PRBTN_END_DT
,cam.SEND_ORG_ID AS JOB_SCND_DEPT_ID
,cam.SEND_POS_CD AS JOB_SCND_JOB_GD_CD
,cam.SEND_POS_GRD_CD AS JOB_SCND_JOB_RANK_CD
,cam.SEND_JOB_CD AS JOB_SCND_JOB_CD
,cam.SEND_DUTY_CD AS JOB_SCND_JOB_TTLE_CD
,cam.REN_YN AS LAYOFF_YN
,COALESCE(mre.dtl_nm ,cls.CD_NM , null) AS CAM_CLASS_NM
 ,COALESCE(mre.dtl_cd ,'EE9999') AS RETRD_TYPE
  ,case when COALESCE(mre.dtl_nm , 'No Code') = 'Resignation(Voluntary)' then 'Voluntary'
       when COALESCE(mre.dtl_nm ,'No Code') = 'Resignation(Involuntary)' then 'Involuntary'
  else null
  end RETRD_TYPE_NM
FROM brz.system_b.CAM_HISTORY cam 
LEFT JOIN brz.system_b.common_code cls ON cls.CD_KIND ='CAM_CAU_CD' and cam.CAU_CD = cls.cd -- Appointment Classification Code
left join ref.hr.code_master mre ON mre.df_cd = 'VolRetCl' AND cam.CAU_CD = mre.Value1 -- Appointment classification from common code
LEFT JOIN brz.system_b.common_code retire on retire.CD_KIND ='CAM_RETIRE_CAU_CD' and cam.RETIRE_TYPE_CD = retire.CD
left join common_schema.apps.m_com_cm_cd_mn com on com.DF_CD = 'ComCl' and cam.SUB_COMPANY_CD = com.Value3 -- Company Name
LEFT JOIN brz.system_b.common_code duty on duty.CD_KIND ='PHM_DUTY_CD' and cam.DUTY_CD = duty.CD -- Title Code (Not Common)
LEFT JOIN brz.system_b.common_code rank on rank.CD_KIND ='PHM_POS_CD' and cam.POS_CD = rank.CD -- Rank Code (Not Common)


UNION ALL
-- System B Manual Appointment Part 
SELECT DISTINCT  
NULL AS LAYOFF_RTRN_SCHDL_DT,  
NULL AS DSPTCH_YN,  
NULL AS DSPTCH_DIV_CD,  
NULL AS DSPTCH_RTRN_SCHDL_DT,  
NULL AS DSPTCH_DEPT_CD,  
NULL AS DSPTCH_PLC_NM,  
NULL AS HIRED_DIV_CD,  
NULL AS PENAL_CNCL_SCHDL_DT,  
NULL AS HR_RFLCTN_YN,  
NULL AS SLRY_JOB_TTLE_GRD,  
NULL AS PRNT_YN,  
NULL AS JOB_GRP_CD,  
NULL AS RFRNC_DIV_CD,  
NULL AS ENTRST_YN,  
NULL AS ENTRST_END_SCHDL_DT,  
NULL AS DEL_YN,  
NULL AS CNF_DOC_NO,  
NULL AS CNF_DOC_NM,  
NULL AS CAM_RSN,  
BEF.note AS NOTE,  
BEF.mod_user_id AS MOD_PRSN,  
BEF.mod_date AS MOD_DTT,  
BEF.tz_cd AS TZ_CD,  
BEF.tz_date AS TZ_DTT,  
null as CMP_DIV_CD,
BEF.sub_company_nm AS CMP_DIV_NM,  
NULL AS SLRY_STRT_SLRY_GRD_CD,  
NULL AS FIRST_APPT_SLRY_KIND_CD,  
NULL AS YRLY_SLRY_AMT,  
NULL AS JOB_SCND_YN,  
NULL AS JOB_SCND_CAM_CMP_CD,  
NULL AS JOB_SCND_JOB_GRP_CD,  
NULL AS RETRD_RSN_CD,  
NULL AS RETRD_RSN_NM,  
NULL AS SLRY_GRD,  
NULL AS SLRY_JOB_GRP,  
NULL AS NEXTERM_JOB_GD_PRMT_DT,  
NULL AS CAM_HIST_ID,  
cast(BEF.emp_id AS STRING) AS EMP_ID,  
NULL AS PRSNL_ID,  
NULL AS CAM_CNF_ID,  
NULL AS CAM_STRT_DT,  
NULL AS CAM_END_DT,  
BEF.sta_ymd AS CAM_DT,  
NULL AS CAM_ORDER,  
NULL AS CAM_CD,  
COALESCE(mre.dtl_cd ,BEF.CAU_CD , null) AS CAM_CLASS_CD,
NULL AS CAM_HR_ZONE_CD,  
NULL AS CAM_DEPT_ID,  
NULL AS CAM_JOB_RANK_CD,  
BEF.pos_nm AS CAM_JOB_RANK_NM,  
NULL AS JOB_CD,  
NULL AS CAM_JOB_TTLE_CD,  
BEF.duty_nm AS CAM_JOB_TTLE_NM,  
NULL AS CAM_EMP_KIND_CD,  
NULL AS CAM_YRLY_SLRY_TYPE,  
NULL AS PRBTN_YN,  
NULL AS PRBTN_END_DT,  
NULL AS JOB_SCND_DEPT_ID,  
NULL AS JOB_SCND_JOB_GD_CD,  
NULL AS JOB_SCND_JOB_RANK_CD,  
NULL AS JOB_SCND_JOB_CD,  
NULL AS JOB_SCND_JOB_TTLE_CD,  
NULL AS LAYOFF_YN,  
COALESCE(mre.dtl_nm ,cls.CD_NM , null) AS CAM_CLASS_NM,
COALESCE(mre.dtl_cd , 'EE9999') AS RETRD_TYPE,
case when COALESCE(mre.dtl_nm , null) = 'Resignation(Voluntary)' then 'Voluntary'
      when COALESCE(mre.dtl_nm , null) = 'Resignation(Involuntary)' then 'Involuntary'
  else 'No Code'  end RETRD_TYPE_NM
FROM BRZ.system_b.cam_history_bef BEF  
LEFT JOIN brz.system_b.common_code cls ON cls.CD_KIND ='CAM_CAU_CD' and BEF.CAU_CD = cls.cd -- Appointment Classification Code
left join ref.hr.code_master mre ON mre.df_cd = 'VolRetCl' AND BEF.CAU_CD = mre.Value1 -- Appointment classification from common code

union all -- System A Appointment

SELECT
NULL AS LAYOFF_RTRN_SCHDL_DT,   
NULL AS DSPTCH_YN,   
NULL AS DSPTCH_DIV_CD, 
NULL AS DSPTCH_RTRN_SCHDL_DT, 
NULL AS DSPTCH_DEPT_CD,  
NULL AS DSPTCH_PLC_NM, 
NULL AS HIRED_DIV_CD,  
NULL AS PENAL_CNCL_SCHDL_DT,  
NULL AS HR_RFLCTN_YN,  
NULL AS SLRY_JOB_TTLE_GRD,  
NULL AS PRNT_YN,  
NULL AS JOB_GRP_CD,  
NULL AS RFRNC_DIV_CD,   
NULL AS ENTRST_YN,   
NULL AS ENTRST_END_SCHDL_DT,   
NULL AS DEL_YN,   
NULL AS CNF_DOC_NO,  
NULL AS CNF_DOC_NM, 
NULL AS CAM_RSN,
past.RMK_DC as NOTE, -- Remarks
NULL AS MOD_PRSN,
NULL AS MOD_DTT,
NULL AS TZ_CD,   
NULL AS TZ_DTT, 
case when past.COMPANY_CD = '1000' then '1f'
     when past.COMPANY_CD = '2000' then '1q'
     when past.COMPANY_CD = '3000' then '1a'
 else past.COMPANY_CD
    END CMP_DIV_CD, -- Company Classification Code 
case when past.COMPANY_CD = '2000' then BIZAREA_NM
  else BIZAREA_NM  end  CMP_DIV_NM, -- Company Classification Name
NULL AS SLRY_STRT_SLRY_GRD_CD,  
NULL AS FIRST_APPT_SLRY_KIND_CD,   
NULL AS YRLY_SLRY_AMT,  
NULL AS JOB_SCND_YN, 
NULL AS JOB_SCND_CAM_CMP_CD, 
NULL AS JOB_SCND_JOB_GRP_CD,  
NULL AS RETRD_RSN_CD, 
NULL AS RETRD_RSN_NM,  
NULL AS SLRY_GRD, 
NULL AS SLRY_JOB_GRP,  
NULL AS NEXTERM_JOB_GD_PRMT_DT,  
NULL AS CAM_HIST_ID,
CASE    
    WHEN past.COMPANY_CD = '3000' THEN 'a' || past.EMP_NO -- Company A
    WHEN past.COMPANY_CD = '2000' THEN 'q' || past.EMP_NO -- Company B
    WHEN past.COMPANY_CD = '1000' THEN 'f' || past.EMP_NO -- Company C
    ELSE past.EMP_NO    
END AS EMP_ID, -- Employee
NULL AS PRSNL_ID,  
NULL AS CAM_CNF_ID,  
NULL AS CAM_STRT_DT,   
NULL AS CAM_END_DT,
to_timestamp(past.GNFD_DT , 'yyyyMMdd') AS CAM_DT, -- Appointment Date 
NULL AS CAM_ORDER,   
NULL AS CAM_CD,
NULL AS CAM_CLASS_CD,
NULL AS CAM_HR_ZONE_CD,   
NULL AS CAM_DEPT_ID,  
NULL AS CAM_JOB_RANK_CD,
past.POSI_NM as CAM_JOB_RANK_NM,  -- Rank Name
NULL AS CAM_JOB_TTLE_CD, 
NULL AS JOB_CD, 
past.ODTY_NM as CAM_JOB_TTLE_NM, -- Title Name
NULL AS CAM_EMP_KIND_CD, 
NULL AS CAM_YRLY_SLRY_TYPE,   
NULL AS PRBTN_YN, 
NULL AS PRBTN_END_DT,  
NULL AS JOB_SCND_DEPT_ID,  
NULL AS JOB_SCND_JOB_GD_CD, 
NULL AS JOB_SCND_JOB_RANK_CD, 
NULL AS JOB_SCND_JOB_CD, 
NULL AS JOB_SCND_JOB_TTLE_CD, 
NULL AS LAYOFF_YN,
COALESCE(dire.DTL_NM, dore.DTL_NM, stre.DTL_NM, past.HR_CD_NM) AS CAM_CLASS_NM, -- Appointment Classification Name 
case when mcls.MNG_DC = '1' then 'VolRetCl1'
     when mcls.MNG_DC = '2' then 'VolRetCl2'
     else 'EE9999' end RETRD_TYPE,
case when mcls.MNG_DC = '1' then 'Voluntary'
     when mcls.MNG_DC = '2' then 'Involuntary'
    else 'No Code'
    end RETRD_TYPE_NM
from slv.system_a.cam_past past
LEFT JOIN (  
  SELECT A.EMP_NO, A.MCLS_CD, A.COMPANY_CD, A.MNG_DC, RETR_DT  
  FROM slv.system_a.HR_EMP_SDTL A  
  INNER JOIN slv.system_a.HR_EMP_MST B ON A.COMPANY_CD = B.COMPANY_CD AND A.EMP_NO = B.EMP_NO  
  INNER JOIN slv.system_a.HR_EMPINFO_DTL HED ON HED.COMPANY_CD = b.COMPANY_CD AND HED.EMP_NO = b.EMP_NO  
  WHERE A.COMPANY_CD IN ('1000','2000','3000')  
  AND (  
    (A.COMPANY_CD = '1000' AND MCLS_CD = 'HR002')  
    OR (A.COMPANY_CD = '2000' AND MCLS_CD = 'HR002')  
    OR (A.COMPANY_CD = '3000' AND MCLS_CD = 'HR002')  
  )  
  AND RETR_DT IS NOT NULL   --- Voluntary/Involuntary Master
) mcls ON past.COMPANY_CD = mcls.COMPANY_CD AND past.EMP_NO = mcls.EMP_NO AND TO_CHAR(TO_TIMESTAMP(past.GNFD_DT , 'yyyyMMdd'), 'yyyyMMdd') = mcls.RETR_DT 
left join ref.hr.code_master dire on dire.df_cd = 'VolRetCl' AND dire.Value3 IS NOT NULL and dire.Value3 = mcls.MNG_DC  -- Change Appointment Classification Name by Common Code
left join ref.hr.code_master dore on dore.df_cd = 'VolRetCl' AND dore.Value5 IS NOT NULL and dore.Value5 = mcls.MNG_DC  -- Change Appointment Classification Name by Common Code
left join ref.hr.code_master stre on stre.df_cd = 'VolRetCl' AND stre.Value7 IS NOT NULL and stre.Value7 = mcls.MNG_DC  -- Change Appointment Classification Name by Common Code
""")

In [ ]:
DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_appoint_history");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_appoint_history
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_appoint_history';